# ML-02 — Research Question and Provisional Lane

Lane 4: CTR / Engagement Opportunity Scoring.

Week 1's Discovery B showed CTR falls in a clean, monotonic step down by position_tier (page_1 highest, deep lowest) — a real, position-adjusted pattern I can use as an expected-CTR baseline. It also showed that comparing CTR across content_type inside a tier breaks down: two of three types have almost no rows in top_3 (n=1, n=5), so any content-type claim there would be noise dressed up as a finding. Lane 4 lets me use the part of the pattern that's real (position tier) and route around the part that isn't (thin content-type slices), rather than forcing a comparison the data can't support yet.

Lane 2 (Refresh Scoring) was the other candidate — the starter proxy label (is_declining_label) is sitting right in the columns. I'm setting it aside for now because I don't yet have evidence from my own analysis that a decline signal is real and not just trend_direction noise, whereas the CTR-by-tier pattern is something I've already checked and trust.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
!git clone https://github.com/jasleen13/ML-Internship.git

df = pd.read_csv("/content/ML-Internship/data/raw/content_refresh_anonymized.csv")

# Evidence for the lane choice: CTR by position tier, with a volume floor so
# single-digit-impression pages don't dominate the mean (flyrank-data skill: ctr is already
# clicks/impressions x 100, so these are percentages, e.g. 0.35 = 0.35% CTR)
floor = df[df["impressions_90d"] >= 100]
tier_ctr = floor.groupby("position_tier")["ctr"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print(tier_ctr)

fatal: destination path 'ML-Internship' already exists and is not an empty directory.
                   mean  count
position_tier                 
page_1         0.354760   8633
top_3          0.334128    533
striking       0.255782   5903
page_3_5       0.142359   6058
deep           0.055415    879


## 2. The question: decision, action, cost of a wrong call

Question: Among visible pages (enough impressions to matter), which ones get meaningfully fewer clicks than other pages at the same position tier?

Unit of analysis: one page (content_id).

Decision it improves: which pages an SEO editor reviews first for a title/meta/snippet rewrite, out of thousands of ranked pages, given limited review time.

Who acts, and how: an SEO content editor opens the top of a ranked "CTR review queue" and rewrites the title tag / meta description / snippet for the pages at the top first.

Output: a ranked list of pages with a CTR gap score (observed CTR vs. that page's position-tier expected CTR), plus impression volume and position, so a reviewer can sanity-check the score before acting.

Cost of a wrong call: the editor spends review time on a page whose low CTR is actually volume noise, not a real title/snippet problem — the feedly article (n=5) and comparison article (n=1) slices from Discovery B are a live example of exactly this trap: a tiny sample can look like a strong signal and be worthless. A minimum-volume floor is therefore part of the method, not an afterthought.

Why data/ML helps here rather than a plain rule: a single global CTR threshold ("flag anything under X%") ignores position — a page ranking #1 with 5% CTR and a page ranking #40 with 5% CTR are not the same story. The pattern (expected CTR depends on position, and the gap from that expectation is what matters) is real but has enough structure that a position-tier-adjusted score does meaningfully better than an unadjusted rule, worth checking with data rather than assuming.

In [12]:
# How big is the candidate pool this decision would actually draw from?
# (pages with enough impressions to trust their CTR, and a real position reading)
candidate_pool = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)]
print(f"Candidate pool: {len(candidate_pool):,} of {len(df):,} pages "
      f"({len(candidate_pool)/len(df):.0%}) pass the volume + position floor")
print(candidate_pool["position_tier"].value_counts())


Candidate pool: 22,006 of 30,000 pages (73%) pass the volume + position floor
position_tier
page_1      8633
page_3_5    6058
striking    5903
deep         879
top_3        533
Name: count, dtype: int64


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*
- **CTR drops steadily by position tier**, from **0.35%** at `page_1` down to **0.055%** at
  `deep` (volume-floored at impressions_90d >= 100, n=879 to n=8,633 per tier) This is the
  expected-CTR-by-tier baseline the whole lane is built on.
- **`avg_position == 0` means "no position data," not rank zero**, in 1,205 rows (about 4% of
  the dataset) These have to be excluded from any position-tier comparison, not treated as
  a page ranking #0.
- **Content-type CTR comparisons inside a tier are unreliable at low volume**: within `top_3`,
  `feedly article` (n=5) and `comparison article` (n=1) aren't trustworthy sample sizes, while
  `keyword article` (n=527) is a reminder that this project needs a volume floor built into
  every group comparison, not just the top-line tier numbers.

In [13]:

# Number 1: CTR by position tier (the core pattern this lane scores against)
tier_ctr = df[df["impressions_90d"] >= 100].groupby("position_tier")["ctr"].agg(["mean", "count"])
print("CTR by position tier (volume-floored):")
print(tier_ctr.sort_values("mean", ascending=False))
print()

# Number 2: avg_position == 0 rows that must be excluded from tier comparisons
no_position = (df["avg_position"] == 0).sum()
print(f"Rows with avg_position == 0 ('no data', not rank zero): {no_position:,} "
      f"({no_position/len(df):.1%} of dataset)")
print()

# Number 3: sample sizes behind a content-type x tier cut, showing where trust breaks down
top3 = df[(df["impressions_90d"] >= 100) & (df["position_tier"] == "top_3")]
print("content_type sample sizes within top_3 (volume-floored):")
print(top3.groupby("content_type")["ctr"].agg(["mean", "count"]).sort_values("count"))


CTR by position tier (volume-floored):
                   mean  count
position_tier                 
page_1         0.354760   8633
top_3          0.334128    533
striking       0.255782   5903
page_3_5       0.142359   6058
deep           0.055415    879

Rows with avg_position == 0 ('no data', not rank zero): 1,205 (4.0% of dataset)

content_type sample sizes within top_3 (volume-floored):
                        mean  count
content_type                       
comparison article  0.000000      1
feedly article      2.900000      5
keyword article     0.310417    527


What this project CAN say:

Observed, position-adjusted CTR patterns: "pages in X tier under-capture clicks relative to other pages in the same tier", an association, backed by a volume floor.
Directional, decision-support recommendations: "these pages are good candidates to review first," ranked by evidence, not "these pages are guaranteed to improve."
Where the data itself is too thin to say anything (e.g. feedly article in top_3, n=5), saying "inconclusive, insufficient sample" is itself a valid, honest result.
What this project CANNOT say:

That rewriting a title/meta/snippet caused any observed CTR change that needs a causal design (an actual before/after experiment), not this dataset.
That a CTR gap means the content is bad, or that a specific search-engine ranking factor is at fault, CTR is one observable signal, not a diagnosis of why.
Anything about Google's algorithm, or that this data proves how search engines behave, this data is FlyRank's observable signals, not Google's internals.
Any claim built on a group with too few rows to be signal rather than noise (per the content-type sample sizes above) those get flagged as inconclusive, not rounded up to a finding.

In [14]:
# ctr can legitimately run well above 1 here, since it's stored as clicks/impressions x 100
# (a percentage) rather than a 0-1 ratio -- e.g. 2.90 means 2.90%, not "290% clicks."
# High values are rare and volume-driven, worth a floor before trusting them, but not a bug.
print("Rows with ctr > 1 (i.e. > 1%):", (df["ctr"] > 1).sum(), "of", len(df))
print("Max ctr in dataset:", df["ctr"].max(), "-> i.e.", f"{df['ctr'].max()}%")

Rows with ctr > 1 (i.e. > 1%): 1689 of 30000
Max ctr in dataset: 100.0 -> i.e. 100.0%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.